# TRIAGE on Kaggle — single NVIDIA T4 (16 GB)

Primary judge: Microsoft Phi-4 14B, loaded locally with 4-bit NF4 quantization on the T4.
Generator: llama3.1:8b through Ollama on CPU so it cannot compete with Phi-4 for GPU memory.
No paid LLM API keys are required.

Before running, Notebook Settings:
- Accelerator: GPU T4
- Internet: ON (required to download model weights and dependencies)

Phi-4 has a 16K native context window. This notebook starts with a 4,096-token judge input budget and batch size 1 for T4 stability. GPU use is sequential: Ollama generation is CPU-only; Phi-4 is the primary model on the GPU.

Run the pilot before treating any result as final. The full benchmark is configured for 50 paper-protocol questions and checkpoints completed questions. Kaggle sessions can time out; retain prepared-artifact and benchmark checkpoint files between sessions.

ClinicalRewardModel is not loaded in the primary run. It is a separate supplementary evaluator and must not replace the Hyper-RAG evaluation dimensions.

In [ ]:
# 1. Clone the correct repository and verify the checked-out revision.
import sys, subprocess, os, json, shutil
print(sys.version)
if sys.version_info >= (3, 13):
    print('Python 3.13 session detected - cell 2 will build a Python 3.12 side-env')

def run_checked(args, label):
    print(f'Running: {label}')
    return subprocess.run(args, check=True)

REPO_URL = 'https://github.com/eklakhdewan/TRIAGE.git'
repo_dir = '/kaggle/working/TRIAGE'
if os.path.isdir(os.path.join(repo_dir, '.git')):
    _remote = subprocess.run(
        ['git', '-C', repo_dir, 'remote', 'get-url', 'origin'],
        capture_output=True, text=True, check=False
    )
    if _remote.returncode != 0 or _remote.stdout.strip().rstrip('/') != REPO_URL.rstrip('/'):
        print('Existing TRIAGE checkout points to a different repository; replacing it.')
        shutil.rmtree(repo_dir)
if not os.path.isdir(os.path.join(repo_dir, '.git')):
    if os.path.exists(repo_dir):
        shutil.rmtree(repo_dir)
    run_checked(['git', 'clone', '--depth', '1', REPO_URL, repo_dir],
                'clone eklakhdewan/TRIAGE')
%cd /kaggle/working/TRIAGE
run_checked(['git', 'pull', '--rebase'], 'update repository')
run_checked(['ls', 'src'], 'verify source tree')
_head_result = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                              capture_output=True, text=True, check=True)
_head = _head_result.stdout.strip()
print('repo HEAD:', _head)
_pin = subprocess.run(
    ['grep', '-q', 'murmurhash==1.0.12', 'requirements.txt'], check=False)
if _pin.returncode != 0:
    raise RuntimeError('Repository is missing the pinned Kaggle environment requirements.')
print('Repository and environment pins verified.')

In [ ]:
# 2. Install dependencies (10-20 min)
# Python 3.13 sessions build a Python 3.12 side-env; all later stages use PY.
import sys
PY = sys.executable
if sys.version_info >= (3, 13):
    run_checked(['bash', '-lc',
                 'set -o pipefail; curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest '
                 '| tar -xj -C /usr/local bin/micromamba'], 'install micromamba')
    run_checked(['micromamba', 'create', '-y', '-p', '/kaggle/tmp/py312', 'python=3.12'],
                'create Python 3.12 side-env')
    PY = '/kaggle/tmp/py312/bin/python'
run_checked([PY, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
            'install pinned project dependencies')
run_checked([PY, '-m', 'pip', 'install', '-q', 'murmurhash==1.0.12'],
            'pin murmurhash (tokenizer specials regression guard)')
run_checked([PY, '-m', 'pip', 'install', '-q', 'numpy==2.5.4'],
            'install compatible NumPy')
run_checked([PY, '-m', 'pip', 'install', '-q', '--no-deps',
             'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz'],
            'install BC5CDR model')
run_checked([PY, '-m', 'pip', 'install', '-q', '--no-deps',
             'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz'],
            'install JNLPBA model')
print('python:', PY)
run_checked([PY, '-c',
             'import chromadb, spacy, scispacy; '
             'print("stack OK:", chromadb.__version__, spacy.__version__, scispacy.__version__); '
             'from importlib.metadata import version as _v; import numpy; '
             'print("low-level:", _v("murmurhash"), numpy.__version__); '
             'assert chromadb.__version__ == "1.5.9"; '
             'assert spacy.__version__ == "3.8.11"'],
            'verify target Chroma and spaCy versions')


In [ ]:
# 2b. Verify the single Kaggle T4 CUDA runtime.
import shutil
if not shutil.which("nvidia-smi"):
    raise RuntimeError(
        "No GPU detected. In Kaggle Notebook Settings, set Accelerator to GPU T4, "
        "restart the session, and run this notebook again."
    )
run_checked(["nvidia-smi"], "verify Kaggle T4")
_cuda_check = subprocess.run(
    [PY, "-c", "import torch; print(torch.cuda.is_available()); "
     "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"],
    capture_output=True, text=True, check=True)
print(_cuda_check.stdout.strip())
if not _cuda_check.stdout.splitlines()[0].strip().endswith("True"):
    raise RuntimeError("PyTorch cannot access CUDA. Fix the Kaggle GPU runtime before proceeding.")
if "T4" not in _cuda_check.stdout:
    print("WARNING: This notebook was sized for a T4; detected GPU differs.")

In [ ]:
# 3. Install Phi-4 quantization support and start the generator on CPU.
run_checked(
    [PY, '-m', 'pip', 'install', '-q',
     'transformers>=4.48,<5.0', 'accelerate>=0.34', 'bitsandbytes>=0.43.3'],
    'install Phi-4 Transformers and 4-bit quantization dependencies'
)

run_checked(['apt-get', '-qq', 'install', '-y', 'zstd'], 'install zstd')
run_checked(
    ['bash', '-lc', 'set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh'],
    'install Ollama'
)

import os, time, subprocess
# Reserve the T4 for Phi-4. Hide CUDA from the Ollama server process.
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'
get_ipython().system_raw(
    'CUDA_VISIBLE_DEVICES="" OLLAMA_KEEP_ALIVE=-1 ollama serve > /tmp/ollama.log 2>&1 &'
)
time.sleep(5)
run_checked(['ollama', 'pull', 'llama3.1:8b'], 'pull HCAD answer generator')
run_checked(['ollama', 'list'], 'verify generator model')
print('Generator: llama3.1:8b (Ollama CPU only)')
print('Judge: microsoft/phi-4 (Transformers 4-bit NF4 on Kaggle T4)')

In [ ]:
# 4. Configure the benchmark judge and optional telemetry secrets.
# Add WANDB_API_KEY and TYPESAFE_API_KEY through Kaggle Secrets if required.
WANDB_API_KEY = ""
TYPESAFE_API_KEY = ""

def _secret(label):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        return ""

WANDB_API_KEY = _secret("WANDB_API_KEY") or WANDB_API_KEY
TYPESAFE_API_KEY = _secret("TYPESAFE_API_KEY") or TYPESAFE_API_KEY

with open('.env', 'w') as f:
    f.write("LLM_PROVIDER=ollama\n")
    f.write("JUDGE_PROVIDER=transformers\n")
    f.write("JUDGE_MODEL=microsoft/phi-4\n")
    f.write("PHI4_LOAD_IN_4BIT=1\n")
    f.write("PHI4_MAX_INPUT_TOKENS=4096\n")
    f.write("OLLAMA_KEEP_ALIVE=-1\n")
    if WANDB_API_KEY:
        f.write(f"WANDB_API_KEY={WANDB_API_KEY}\n")
    else:
        f.write("WANDB_MODE=disabled\n")
    if TYPESAFE_API_KEY:
        f.write(f"TYPESAFE_API_KEY={TYPESAFE_API_KEY}\n")

print("Judge configuration written:")
for line in open('.env').read().splitlines():
    print(line.split('=', 1)[0] + '=***')

# Smoke-test the repository judge adapter in a child process. The process exits
# afterward, releasing Phi-4 GPU memory before the benchmark starts.
_smoke = (
    "from src.config import JUDGE_HF_MODEL; "
    "from src.core.llm_client import UnifiedLLMClient; "
    "j=UnifiedLLMClient(model=JUDGE_HF_MODEL, temperature=0.0, max_tokens=32); "
    "print(j.chat([{'role':'user','content':'Reply with exactly: PHI4_READY'}], max_tokens=32))"
)
run_checked([PY, '-c', _smoke], 'load Phi-4 4-bit and run judge smoke test')

### Optional: restore artifacts from a previous session

Skip this on a first run. After saving artifacts (bottom of this notebook) as a private Kaggle dataset, add that dataset in the notebook's right panel ("+ Add Input"), then run the cell to skip the 2–5 h `prepare --rebuild`.

In [ ]:
# 4b. RESTORE (only when the artifacts dataset is attached).
# Restore by archive, extracted store, or legacy flat layout; verify all required files.
RESTORE = True
RESTORED = False
if RESTORE:
    import glob, os, shutil
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('data/vectordb', exist_ok=True)
    tarballs = sorted(glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)
                      + glob.glob('/kaggle/input/**/*.tgz', recursive=True))
    zips = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
    if tarballs:
        print(f'Restoring from {tarballs[0]} ...')
        run_checked(['tar', 'xzf', tarballs[0]], 'extract restore archive')
    elif zips:
        print(f'Restoring from {zips[0]} ...')
        run_checked(['unzip', '-q', '-o', zips[0]], 'extract zip archive')
    else:
        chroma_hits = glob.glob('/kaggle/input/**/chroma.sqlite3', recursive=True)
        if chroma_hits:
            src_vdb = os.path.dirname(chroma_hits[0])
            print(f'Restoring extracted store from {src_vdb} ...')
            shutil.copytree(src_vdb, 'data/vectordb', dirs_exist_ok=True)
            for fname in ('primekg_graph.gpickle', 'sapbert_node_index.faiss',
                          'sapbert_node_names.json'):
                hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
                if hits:
                    shutil.copy2(hits[0], f'data/processed/{fname}')
                    print(f'  + {fname}')
        else:
            legacy = '/kaggle/input/triage-artifacts'
            if os.path.isdir(legacy):
                shutil.copytree(legacy, 'data/processed', dirs_exist_ok=True)
                legacy_vdb = os.path.join(legacy, 'vectordb')
                if os.path.isdir(legacy_vdb):
                    shutil.copytree(legacy_vdb, 'data/vectordb', dirs_exist_ok=True)

    # Check if files were extracted or mounted anywhere under /kaggle/input and copy if missing
    chroma = 'data/vectordb/chroma.sqlite3'
    gpickle = 'data/processed/primekg_graph.gpickle'
    index = 'data/processed/sapbert_node_index.faiss'
    if not (os.path.exists(chroma) and os.path.exists(gpickle) and os.path.exists(index)):
        c_hits = glob.glob('/kaggle/input/**/chroma.sqlite3', recursive=True)
        if c_hits and not os.path.exists(chroma):
            shutil.copytree(os.path.dirname(c_hits[0]), 'data/vectordb', dirs_exist_ok=True)
        for fname in ('primekg_graph.gpickle', 'sapbert_node_index.faiss', 'sapbert_node_names.json'):
            target = f'data/processed/{fname}'
            if not os.path.exists(target):
                f_hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
                if f_hits:
                    shutil.copy2(f_hits[0], target)

    if (os.path.exists(chroma) and os.path.getsize(chroma) > 10_000_000
            and os.path.exists(gpickle) and os.path.exists(index)):
        RESTORED = True
        print(f'Restore OK: chroma.sqlite3 = {os.path.getsize(chroma)/1e6:.0f} MB')
        run_checked(['ls', '-la', 'data/processed/', 'data/vectordb/'],
                    'verify restored artifacts')
    else:
        print('No complete pre-built artifacts found in /kaggle/input (or restore was skipped).')
        print('Listing of /kaggle/input:')
        subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], check=False)
        print('RESTORED = False. Stage 6 (cell 9) will prepare the knowledge graph and vector store from scratch.')
        RESTORED = False
        run_checked(['echo', 'Artifact restore skipped; cell 6 will build from scratch.'],
                    'log restore skip')


In [ ]:
# 5. Full test suite (run only in this pinned Kaggle environment).
run_checked([PY, '-m', 'pytest', 'tests', '-q'], 'full pytest suite')


In [ ]:
# 6. Prepare the knowledge graph and vector store (the multi-hour stage).
if RESTORED:
    print('Skipped prepare: artifacts restored in cell 4b.')
    run_checked(['du', '-sh', 'data/vectordb', 'data/processed'],
                'verify restored store sizes')
else:
    import os
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ.setdefault('PMC_NUM_ARTICLES', '1000')
    run_checked([PY, 'src/main.py', 'prepare', '--rebuild'], 'prepare graph and vector store')
    for required in ('data/vectordb/chroma.sqlite3',
                     'data/processed/primekg_graph.gpickle',
                     'data/processed/sapbert_node_index.faiss'):
        if not os.path.isfile(required) or os.path.getsize(required) == 0:
            raise RuntimeError(f'Prepare completed without required output: {required}')


In [ ]:
# 6b. Snapshot prepared artifacts before the benchmark.
_archive = '/kaggle/working/triage_artifacts.tar.gz'
run_checked(['tar', 'czf', _archive,
             'data/processed/primekg_graph.gpickle',
             'data/processed/sapbert_node_index.faiss',
             'data/processed/sapbert_node_names.json', 'data/vectordb'],
            'snapshot prepared artifacts')
if not os.path.isfile(_archive) or os.path.getsize(_archive) <= 10_000_000:
    raise RuntimeError('Artifact archive is missing or unexpectedly small.')
run_checked(['du', '-sh', _archive], 'verify artifact archive')


In [ ]:
# 7. Smoke test one query through all three systems.
run_checked([PY, 'src/main.py', 'demo'], 'run three-system smoke test')


In [ ]:
# 8. Optional session-1 pilot.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
RUN_PILOT = False
if RUN_PILOT:
    _pilot_path = 'results/base_paper_benchmark_results.json'
    _pilot_before = os.stat(_pilot_path).st_mtime_ns if os.path.exists(_pilot_path) else None
    run_checked([PY, 'src/main.py', 'base-paper', '--samples', '24', '--plot'],
                'run 24-question pilot')
    if not os.path.exists(_pilot_path) or (
            _pilot_before is not None and os.stat(_pilot_path).st_mtime_ns <= _pilot_before):
        raise RuntimeError('Pilot exited without producing a fresh results file.')
else:
    print('Skipped pilot (RUN_PILOT=False).')


In [ ]:
# 9a. Train the learned query-adaptive tensor on the dev pool.
TRAIN_TENSOR = True  # Re-enabled: Phase-2 listwise ranking loss and SciSpacy tokenizer recovery operational
if TRAIN_TENSOR:
    _tensor_path = 'models/dynamic_tensor.pt'
    _tensor_before = os.stat(_tensor_path).st_mtime_ns if os.path.exists(_tensor_path) else None
    run_checked([PY, 'src/main.py', 'train-tensor', '--samples', '60', '--epochs', '3'],
                'train dev-pool tensor')
    if not os.path.exists(_tensor_path) or (
            _tensor_before is not None and os.stat(_tensor_path).st_mtime_ns <= _tensor_before):
        raise RuntimeError('Tensor training exited without producing a fresh checkpoint.')
else:
    print('Skipped tensor training (TRAIN_TENSOR=False).')


In [ ]:
# 9c. Deliverable-B dev-pool retrieval eval (Phase 3).
# Paired before/after of channel-presence masking + relation-vocab
# mapping + hybrid dense/structured scoring + Gate-2 saturation cap on
# the DEV pool (retrieval-only, no judge). Calibrates tau_sat (p25 of
# gold dev candidate scores, hybrid scale) and prints the GO/STOP
# verdict (GO unless NDCG@5 drops vs the legacy baseline arm).
# The test pool stays untouched until GO.
import glob, json, os
run_checked([PY, 'src/main.py', 'retrieval-eval', '--samples', '60', '--pool', 'dev'],
            'run Deliverable-B dev-pool retrieval eval')
_eval_files = sorted(glob.glob('results/retrieval_eval_dev_n*.json'))
if not _eval_files:
    raise RuntimeError('Retrieval eval exited without producing results.')
with open(_eval_files[-1], encoding='utf-8') as _f:
    _retrieval_eval = json.load(_f)
if _retrieval_eval.get('go_stop', {}).get('decision') != 'GO':
    print(
        'NOTICE: Deliverable-B GO/STOP gate returned '
        + str(_retrieval_eval.get('go_stop')))
else:
    print('Deliverable-B GO verdict:',
          _retrieval_eval.get('delta_vs_baseline'))


In [ ]:
# 9d. Phase-4a hub-weighting ablation (dev pool, retrieval-only).
# Compares desc (legacy hub-favoring) vs soft (deployed: inverse
# log-degree, NO hard cap) vs asc_cap (hard deg<=cap drop, expected
# to lose canonical receptors like NR3C1). Gold Recall@5/NDCG@5 plus
# canonical-hub retention rate.
run_checked([PY, 'src/main.py', 'hub-eval', '--samples', '60', '--pool', 'dev'],
            'run Phase-4a hub ablation')
import glob as _glob, json as _json
_hub_files = sorted(_glob.glob('results/hub_ablation_dev_n*.json'))
if _hub_files:
    with open(_hub_files[-1], encoding='utf-8') as _f:
        print(_json.dumps(_json.load(_f), indent=2))


In [ ]:
# 9. Main full benchmark: static weights and held-out test questions.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
print('HCAD_EXPECTED_GATE_MODE:', os.environ['HCAD_EXPECTED_GATE_MODE'])
# Deliverable-B: activate the calibrated Gate-2 saturation cap.
import glob as _glob, json as _json
_retrieval_eval_files = sorted(_glob.glob('results/retrieval_eval_dev_n*.json'))
if _retrieval_eval_files:
    with open(_retrieval_eval_files[-1], encoding='utf-8') as _f:
        _tau_sat = _json.load(_f).get('tau_sat')
    if _tau_sat:
        os.environ['HCAD_TAU_SAT'] = str(_tau_sat)
        print('Gate-2 saturation cap active: tau_sat =', _tau_sat)

RUN_FULL_BENCHMARK = True
if RUN_FULL_BENCHMARK:
    _results_path = 'results/base_paper_benchmark_results.json'
    _results_before = os.stat(_results_path).st_mtime_ns if os.path.exists(_results_path) else None
    run_checked([PY, 'src/main.py', 'base-paper', '--samples', '50', '--plot'],
                'run 50-question test benchmark')
    if not os.path.exists(_results_path) or (
            _results_before is not None and os.stat(_results_path).st_mtime_ns <= _results_before):
        raise RuntimeError('Benchmark exited without producing fresh results.')
    with open(_results_path, encoding='utf-8') as _f:
        _benchmark_results = json.load(_f)
        _run_manifest = _benchmark_results.get('run_manifest', {})
    if (_benchmark_results.get('num_samples') != 50
            or _run_manifest.get('question_pool') != 'test'):
        raise RuntimeError('Benchmark result count or question pool does not match this run.')
    if _benchmark_results.get('sample_source_counts') != {'paper_protocol': 50}:
        raise RuntimeError('Benchmark results do not contain all 50 paper-protocol questions.')
else:
    print('Skipped full benchmark (RUN_FULL_BENCHMARK=False).')


In [ ]:
# 9b. Optional learned-tensor ablation on the identical TEST questions.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
RUN_ABLATION = False
if RUN_ABLATION:
    import os
    os.environ['HCAD_TENSOR_MODE'] = 'learned'
    os.environ['BENCHMARK_CHECKPOINT'] = 'base_paper_checkpoint_learned.jsonl'
    os.environ['BENCHMARK_RESULTS_FILE'] = 'base_paper_benchmark_results_learned.json'
    _ablation_path = 'results/base_paper_benchmark_results_learned.json'
    _ablation_before = os.stat(_ablation_path).st_mtime_ns if os.path.exists(_ablation_path) else None
    try:
        run_checked([PY, 'src/main.py', 'base-paper', '--samples', '50', '--plot'],
                    'run learned-tensor ablation')
        if not os.path.exists(_ablation_path) or (
                _ablation_before is not None and os.stat(_ablation_path).st_mtime_ns <= _ablation_before):
            raise RuntimeError('Ablation exited without producing fresh results.')
    finally:
        del os.environ['HCAD_TENSOR_MODE'], os.environ['BENCHMARK_CHECKPOINT'], os.environ['BENCHMARK_RESULTS_FILE']
else:
    print('Skipped learned-tensor ablation (RUN_ABLATION=False).')


In [ ]:
# 10. Inspect the fresh benchmark results.
import json
with open('results/base_paper_benchmark_results.json', encoding='utf-8') as f:
    r = json.load(f)
if r.get('num_samples') != 50 or r.get('run_manifest', {}).get('question_pool') != 'test':
    raise RuntimeError('Results are stale or do not contain the 50-question test pool.')
print(json.dumps({k: r[k] for k in ('scoring_based_assessment', 'retrieval_quality',
      'citation_precision', 'significance_testing')}, indent=2)[:2000])


In [ ]:
# 11. Optional live API check with checked HTTP status and clean shutdown.
import subprocess
_server_log = open('/tmp/serve.log', 'w')
_server = subprocess.Popen(
    [PY, 'src/main.py', 'serve', '--host', '127.0.0.1', '--port', '8000'],
    stdout=_server_log, stderr=subprocess.STDOUT,
)
try:
    run_checked(['curl', '--fail', '--show-error', '--retry', '30',
                 '--retry-connrefused', '--retry-delay', '2',
                 'http://127.0.0.1:8000/api/triage/health'],
                'check API health endpoint')
    run_checked(['curl', '--fail', '--show-error', '-X', 'POST',
                 'http://127.0.0.1:8000/api/triage/query',
                 '-H', 'Content-Type: application/json',
                 '-d', '{"query":"Can Metformin be given to patients with severe renal failure given lactic acidosis risk?","config":{"tauEntropy":0.25}}' ],
                'check clinical query endpoint')
finally:
    _server.terminate()
    _server.wait(timeout=10)
    _server_log.close()


In [ ]:
# 12. Copy results and resume files to the working directory.
import glob, os, shutil
_copy_paths = [
    'results/base_paper_checkpoint.jsonl',
    'results/base_paper_benchmark_results.json',
    'results/base_paper_checkpoint_learned.jsonl',
    'results/base_paper_benchmark_results_learned.json',
    'models/dynamic_tensor.pt',
]
_copy_paths.extend(glob.glob('results/paper_questions_*.json'))
for _path in _copy_paths:
    if os.path.isfile(_path):
        shutil.copy2(_path, '/kaggle/working/')
        print('Copied:', _path)
    elif _path in ('results/base_paper_checkpoint.jsonl',
                   'results/base_paper_benchmark_results.json'):
        raise RuntimeError(f'Required benchmark output is missing: {_path}')
run_checked(['ls', '-la', '/kaggle/working/'], 'verify saved outputs')


## Troubleshooting

- Phi-4 CUDA OOM: confirm Ollama was started with CUDA hidden; lower PHI4_MAX_INPUT_TOKENS from 4096 to 3072. Do not load ClinicalRewardModel in the same process.
- Phi-4 load fails: verify Kaggle GPU T4 is enabled, Internet is ON, and bitsandbytes, accelerate, and transformers>=4.48 installed in the PY environment.
- Slow generation: Ollama is intentionally CPU-only to reserve the T4 for Phi-4. This avoids GPU contention but makes answer generation slower.
- Judge mismatch: check .env contains JUDGE_PROVIDER=transformers and JUDGE_MODEL=microsoft/phi-4. Transformer judge errors must stop the run; do not silently switch judges mid-benchmark.
- Ollama dead after restart: re-run cell 3; the server does not survive kernel restarts.
- Checkpoint reuse: benchmark checkpoint and question cache allow completed questions to be resumed.
- Kaggle session timeout: retain benchmark checkpoint and prepared artifacts. Run preparation and benchmark in separate sessions if necessary; restore the artifacts dataset before the benchmark session.
- ClinicalRewardModel: run only as a separate supplementary evaluation stage after the primary Hyper-RAG-compatible evaluation. It does not replace Phi-4's five scoring dimensions or eight pairwise criteria.